In [1]:
# 1. Oldingi natijalarni o'qish (whisper_baho.ipynb dagi eval_50_natija.csv)
import re
import pandas as pd
import jiwer

import sys, importlib
sys.path.insert(0, "../04_normalize")   # normalize.py shu papkada
import normalize
importlib.reload(normalize)

df = pd.read_csv("eval_50_natija.csv")
print("Kliplar:", len(df))

Kliplar: 50


In [2]:
# 2. Qozoq kirill -> qoraqalpoq lotin jadvali
HARFLAR = {
    "а": "a", "ә": "á", "б": "b", "в": "v", "г": "g", "ғ": "ǵ", "д": "d",
    "е": "e", "ё": "yo", "ж": "j", "з": "z", "и": "i", "й": "y", "к": "k",
    "қ": "q", "л": "l", "м": "m", "н": "n", "ң": "ń", "о": "o", "ө": "ó",
    "п": "p", "р": "r", "с": "s", "т": "t", "ұ": "u", "ү": "ú", "ф": "f",
    "х": "x", "һ": "h", "ц": "c", "ч": "ch", "ш": "sh", "щ": "sh", "ъ": "",
    "ы": "ı", "і": "i", "ь": "", "э": "e", "ю": "yu", "я": "ya",
}
UNLILAR = "аәеёиоөұүыіэюяaáeioóuúı"


def lotinga(matn):
    matn = matn.lower()
    # "у": unlidan keyin -> "w" (жау -> jaw), aks holda -> "u"
    matn = re.sub(rf"(?<=[{UNLILAR}])у", "w", matn)
    matn = matn.replace("у", "u")
    natija = ""
    for harf in matn:
        natija += HARFLAR.get(harf, harf)
    return natija


print(lotinga("мәмлекетіміз басшысын исламабад қаласында"))

mámleketimiz basshısın islamabad qalasında


In [3]:
# 3. kk natijasini lotinga o'girish
df["model_kk_lotin"] = [normalize.kichik_harf(lotinga(str(t))) for t in df["model_kk"]]

for i in range(3):
    print("HAQIQIY :", df.loc[i, "matn_model"])
    print("UZ      :", df.loc[i, "model_uz_norm"])
    print("KK LOTIN:", df.loc[i, "model_kk_lotin"])
    print()

HAQIQIY : mámleketimiz basshısın islamabad qalasında ajıratılǵan rezidenciyada pakistan islam respublikası bas ministriniń orınbasarı sırtqı isler ministri ishoq dor kútip aldı
UZ      : mwen legey edmuz başlisun islam o bot kalasna da ajaratilgarezdiyen tisyaasda da pakistan islam disbublu yukas baslam enstere n o rum bas hara sqasle minsterga ishok dhor kutubol da
KK LOTIN: mámlege edimiz bashtısın islamu but qalasında ajatılǵar ezdein tesisti dida paqistan islamdı spublikast bastımensterin orım basarı sútkıslen misterińi eshok dor kúti boldı

HAQIQIY : búgingi kúnde tórt million gektar egislik maydanı bir pútin onnan bes million tonna sıyımlılıqqa iye saqlaw qoymaxanaları hám úsh pútin onnan bes million tonna ónimdi qayta islew quwatlıqları xalıqtı arzan hám sapalı azıq awqat ónimleri menen támiyinlew sonday aq tórt milliard dollarlıq ónimdi eksport etiw imkaniyatın bermekte
UZ      : bukun deyek törd milion gektar yegislik maydanabriyarun milion donna isseimlalukka iyes hakla uttuk

In [4]:
# 4. Solishtirish
haqiqiy = df["matn_model"].tolist()
variantlar = {
    "uz": "model_uz_norm",
    "kk (kirill)": "model_kk_norm",
    "kk -> lotin": "model_kk_lotin",
}

natija = []
for nom, ustun in variantlar.items():
    taxmin = df[ustun].fillna("").tolist()
    natija.append({
        "variant": nom,
        "WER": round(jiwer.wer(haqiqiy, taxmin) * 100, 1),
        "CER": round(jiwer.cer(haqiqiy, taxmin) * 100, 1),
    })

natija = pd.DataFrame(natija)
print(natija.to_string(index=False))

natija.to_csv("baseline_natija.csv", index=False)
df.to_csv("eval_50_natija.csv", index=False)

    variant   WER  CER
         uz 134.4 58.5
kk (kirill) 108.5 99.1
kk -> lotin 100.8 48.2
